# <a id='toc1_'></a>[Section 3: Modeling](#toc0_)

The goal of this notebook is to run the baseline models and tune hyperparameters for the following models:
- Logistic Regression
- Random Forest
- XGBoost Models

Before running the baseline models, however, I addressed the class imbalance by applying SMOTE to the training data. Prior to handling the class imbalance, the baseline models were severely overfitted.



**Table of contents**<a id='toc0_'></a>    
- [Section 3: Modeling](#toc1_)    
- [1. Set Up](#toc2_)    
- [2. Model Preparation](#toc3_)    
  - [2.1 Train/Test Split](#toc3_1_)    
- [3. Handling the Class Imbalance](#toc4_)    
  - [3.1 Applying SMOTE](#toc4_1_)    
  - [3.2 Scaling](#toc4_1_1_)    
- [4. Baseline Models](#toc5_)    
    - [4.1 Logistic Regression](#toc5_1_1_1_)    
    - [4.2 Random Forest Balanced](#toc5_1_1_2_)    
    - [4.3 XGBoost Balanced](#toc5_1_1_3_)    
- [5. Parameter Tuned Models](#toc6_)    
  - [5.1 Logistic Regression Hyperparameter Tuning](#toc6_1_1_)    
    - [5.1.1 Hyperparameter optimization for C](#toc6_1_1_1_)    
    - [5.1.2 Manual Hyperparameter Logistic Regression Model (log_reg_sm2)](#toc6_1_1_2_)    
    - [5.1.3 PCA with Logistic Regression model](#toc6_1_1_3_)    
  - [5.2 Random Forest Hyperparameter Tuning](#toc6_2_)    
    - [5.2.1 Manual Hyperparameter Optimization Random Forest Model (random_forest_sm2)](#toc6_2_1_)    
    - [5.2.2 Grid Searched HyperParameters Random Forest Model random_forest_sm3](#toc6_2_2_)    
  - [5.3 XGBoost Model with Hyperparameter Tuning](#toc6_3_)    
    - [5.3.1 Grid Searched Hyperparameter XGBoost Model XGBoost_model_sm3](#toc6_3_1_)    
- [6. Summary](#toc7_)    
- [7. Export Data and Models](#toc8_)    
- [8. Appendix](#toc9_)    
  - [8.1 Random Forest Model: Manually identifying best hyperparameters](#toc9_1_1_)    
  - [8.2 Grid Searches](#toc9_2_)    
    - [8.2.1 Random Forest Grid Search for Hyperparameters](#toc9_2_1_)     
    - [8.2.2 XGBoost Model Grid Search for optimial Hyperparameters](#toc9_2_2_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc2_'></a>[1. Set Up](#toc0_)

In [1]:
# Standard imports
import numpy as np
import pandas as pd

# Plotting
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objs as go

# Modeling
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Lasso, Ridge, LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

#Metrics 
from sklearn.metrics import f1_score, precision_score, recall_score, roc_curve, roc_auc_score
from sklearn.metrics import classification_report, confusion_matrix, RocCurveDisplay, ConfusionMatrixDisplay
import shap
from sklearn.inspection import permutation_importance
import lime
import lime.lime_tabular

# Balancing
from imblearn.over_sampling import SMOTE


from tempfile import mkdtemp
import joblib

In [2]:
# show all dataframe columns
pd.set_option('display.max_columns', None)
# set matplotlib global settings eg. figsize
plt.rcParams['figure.figsize'] = (8.0, 6.0)

In [3]:
# Import hotel_clean_df
final_df = pd.read_csv('../data/processed/final_df.csv')


In [4]:
final_df.head(5)

,is_canceled,lead_time,arrival_date_week_number,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,is_repeated_guest,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,day_demand,los,arrival_year,arrival_month,arrival_dow,arrival_day,hotel_City Hotel,hotel_Resort Hotel,market_segment_Aviation,market_segment_Complementary,market_segment_Corporate,market_segment_Direct,market_segment_Groups,market_segment_Offline TA/TO,market_segment_Online TA,market_segment_Undefined,distribution_channel_Corporate,distribution_channel_Direct,distribution_channel_GDS,distribution_channel_TA/TO,distribution_channel_Undefined,deposit_type_No Deposit,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Transient
0,0,342,27,0,0,2,0,0,1,0,0,3,0,0,0,0.0,0,0,122,0,2015,7,2,1,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1
1,0,737,27,0,0,2,0,0,1,0,0,4,0,0,0,0.0,0,0,122,0,2015,7,2,1,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1
2,0,7,27,0,1,1,0,0,1,0,0,0,0,0,0,75.0,0,0,122,1,2015,7,2,1,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1
3,0,13,27,0,1,1,0,0,1,0,0,0,1,0,0,75.0,0,0,122,1,2015,7,2,1,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,1
4,0,14,27,0,2,2,0,0,1,0,0,0,1,0,0,98.0,0,1,122,2,2015,7,2,1,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,1


In [5]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114737 entries, 0 to 114736
Data columns (total 43 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   is_canceled                     114737 non-null  int64  
 1   lead_time                       114737 non-null  int64  
 2   arrival_date_week_number        114737 non-null  int64  
 3   stays_in_weekend_nights         114737 non-null  int64  
 4   stays_in_week_nights            114737 non-null  int64  
 5   adults                          114737 non-null  int64  
 6   children                        114737 non-null  int64  
 7   babies                          114737 non-null  int64  
 8   meal                            114737 non-null  int64  
 9   is_repeated_guest               114737 non-null  int64  
 10  previous_bookings_not_canceled  114737 non-null  int64  
 11  booking_changes                 114737 non-null  int64  
 12  agent           

# <a id='toc3_'></a>[2. Model Preparation](#toc0_)

Before I begin modeling, I will need to perform a train-test split as well as handle the class imbalance. Prior to handling class imbalance, I had run models and the results were severely overfitted.

## <a id='toc3_1_'></a>[2.1 Train/Test Split](#toc0_)

In [6]:
# Separating X and y target variable (is_canceled)
X = final_df.drop('is_canceled', axis=1)
y = final_df['is_canceled']

In [7]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: is_canceled, dtype: int64

In [8]:
y.value_counts()

is_canceled
0    71834
1    42903
Name: count, dtype: int64

In [9]:
# Evaluating the balance of the data set
y.value_counts(normalize=True)*100

is_canceled
0    62.607529
1    37.392471
Name: proportion, dtype: float64

In [10]:
# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=24, stratify=y)

In [11]:
# Checking the size of y_train
np.unique(y_train,return_counts=True)

(array([0, 1]), array([57467, 34322]))

In [12]:
# Checking the size of y_test
np.unique(y_test,return_counts=True)

(array([0, 1]), array([14367,  8581]))

In [13]:
# Sanity Check of X_train
X_train

,lead_time,arrival_date_week_number,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,is_repeated_guest,previous_bookings_not_canceled,booking_changes,agent,company,days_in_waiting_list,adr,required_car_parking_spaces,total_of_special_requests,day_demand,los,arrival_year,arrival_month,arrival_dow,arrival_day,hotel_City Hotel,hotel_Resort Hotel,market_segment_Aviation,market_segment_Complementary,market_segment_Corporate,market_segment_Direct,market_segment_Groups,market_segment_Offline TA/TO,market_segment_Online TA,market_segment_Undefined,distribution_channel_Corporate,distribution_channel_Direct,distribution_channel_GDS,distribution_channel_TA/TO,distribution_channel_Undefined,deposit_type_No Deposit,deposit_type_Non Refund,deposit_type_Refundable,customer_type_Transient
10417,10,14,1,1,1,0,0,1,0,0,0,0,1,0,45.0,0,0,231,2,2017,4,0,3,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,1
69810,104,31,0,4,2,0,0,1,0,0,0,1,0,0,125.0,0,2,150,4,2017,8,2,2,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,1
63046,56,15,2,2,2,1,0,1,0,0,0,1,0,0,144.0,0,0,273,4,2017,4,6,9,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,1
27719,14,43,0,3,1,0,0,1,0,0,0,1,0,0,66.0,0,1,136,3,2016,10,1,18,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,1
10752,63,16,1,3,1,0,0,1,0,0,1,0,1,0,85.0,0,0,158,4,2017,4,3,20,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108565,171,22,0,2,2,0,0,1,0,0,0,1,0,0,126.0,0,0,254,2,2017,6,4,2,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1
88837,88,30,1,2,3,0,0,1,0,0,0,1,0,0,96.9,0,2,184,3,2016,7,0,18,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,1
42765,99,44,2,2,2,0,0,1,0,0,0,1,0,0,65.0,0,0,106,4,2015,10,4,30,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,1
46392,85,15,0,3,2,0,0,1,0,0,1,1,0,0,99.3,0,0,196,3,2016,4,3,7,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,1


# <a id='toc4_'></a>[3. Handling the Class Imbalance](#toc0_)

 The imbalance is 37% class 1 and 63% class 0. Through exploration, I discovered that without imbalancing, there is severe overfitting on the baseline models. Therefore, before scaling the data, I need to handle the class imbalance by applying SMOTE.

## <a id='toc4_1_'></a>[3.1 Applying SMOTE](#toc0_)

1. Instantiate SMOTE sampler, fit it to the training data, then resample the data
2. Scale the sampled train data and the unsampled test data
3. Instantiate and fit to scaled & balanced training data for baseline models
4. Make Predictions and evaluate with new balanced data

In [14]:
# Step 1: Instantiate SMOTE
smote = SMOTE(random_state=24)

# Fit training data and Resample
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

# Check what SMOTE has done
print('Original class distribution')
display(pd.Series(y_train).value_counts().sort_index())

print('\nResampled class distribution')
display(pd.Series(y_train_sm).value_counts().sort_index())

Original class distribution


is_canceled
0    57467
1    34322
Name: count, dtype: int64


Resampled class distribution


is_canceled
0    57467
1    57467
Name: count, dtype: int64

## <a id='toc4_1_1_'></a>[3.2 Scaling](#toc0_)

In [15]:
# Scale the sampled train data and the unsampled test data
ss_sm = StandardScaler().fit(X_train_sm)
X_train_sm_ss = ss_sm.transform(X_train_sm)
X_test_sm_ss = ss_sm.transform(X_test)


# <a id='toc5_'></a>[4. Baseline Models](#toc0_)


## <a id='toc5_1_1_1_'></a>[4.1 Logistic Regression](#toc0_)

In [16]:
# Instantiate the model
log_reg_sm = LogisticRegression()

# Fit the model
log_reg_sm.fit(X_train_sm_ss, y_train_sm)

# predict classification
y_test_pred=log_reg_sm.predict(X_test_sm_ss)

# Score the train set
print(f'Train score: {log_reg_sm.score(X_train_sm_ss, y_train_sm)*100:0.2f}%')

# Evaluate the test set
print(f'Test score: {log_reg_sm.score(X_test_sm_ss, y_test)*100:0.2f}%')
print(f'Precision score: {precision_score(y_test,y_test_pred)*100:0.2f}%')
print(f'Recall score: {recall_score(y_test,y_test_pred)*100:0.2f}%')
print(f'F1 score: {f1_score(y_test,y_test_pred)*100:0.2f}%')

Train score: 79.90%
Test score: 77.97%
Precision score: 72.92%
Recall score: 65.34%
F1 score: 68.92%


/Users/brianlui/anaconda3/envs/rm_system/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


| Model              | Train Score | Test Score | Precision | Recall  | f1-score |
|--------------------|-------------|------------|-----------|---------|----------|
| log_reg_sm         | 79.90%      | 77.97%     | 72.92%    | 65.34%  | 68.92%   |


The baseline logistic regression model is showing a ~78% accuracy score with slight overfitting since the train score is ~80%. The model is providing higher precision than recall with an f1-score of only ~69%.


In [17]:
# Checking coefficients values
pd.DataFrame({'coeff': log_reg_sm.coef_[0]}, index=X_train.columns).sort_values('coeff')

,coeff
stays_in_week_nights,-8.386190
stays_in_weekend_nights,-4.575655
hotel_City Hotel,-3.089460
hotel_Resort Hotel,-3.019346
market_segment_Offline TA/TO,-2.526074
market_segment_Online TA,-2.431157
required_car_parking_spaces,-2.411801
market_segment_Groups,-2.259216
deposit_type_No Deposit,-2.202433
distribution_channel_TA/TO,-2.081598


The highest coefficient is los (length of stay), which was a feature engineered variable created from the preprocessing steps. This feature was calculated by combining "stays in weekend nights" with "stays in weekday nights".  Depending on how the model performs and the overfitting, I may remove those 2 features since they are redundant now that we have length of stay.  For now I will keep it to see if there are any trends that surface for weekend stays vs. weekday stays.

## <a id='toc5_1_1_2_'></a>[4.2 Random Forest Balanced](#toc0_)

In [ ]:
# Instantiate random forest
random_forest_sm = RandomForestClassifier(n_estimators=100) # instantiating and picking to train 100 DT models which is the default

# Fit the random forest model
random_forest_sm.fit(X_train_sm_ss, y_train_sm)

# predict classification
y_test_pred=random_forest_sm.predict(X_test_sm_ss)

# Score the train set
print(f'Train score: {random_forest_sm.score(X_train_sm_ss, y_train_sm)*100:0.2f}%')

# Evaluate the test set
print(f'Test score: {random_forest_sm.score(X_test_sm_ss, y_test)*100:0.2f}%')
print(f'Precision score: {precision_score(y_test,y_test_pred)*100:0.2f}%')
print(f'Recall score: {recall_score(y_test,y_test_pred)*100:0.2f}%')
print(f'F1 score: {f1_score(y_test,y_test_pred)*100:0.2f}%')

| Model              | Train Score | Test Score | Precision | Recall  | f1-score | n_estimators |
|--------------------|-------------|------------|-----------|---------|----------|--------------|
| random_forest_sm   | 99.08%      | 85.75%     | 83.57%    | 77.03%  | 80.16%   | 1000         |



The baseline random forest model is showing a 85% accuracy score with overfitting since the train score is 99%. The model is providing higher precision than recall as well, with an f1-score of only 80%.


## <a id='toc5_1_1_3_'></a>[4.3 XGBoost Balanced](#toc0_)

In [ ]:
# Instantiate XGBoost
XGBoost_model_sm = XGBClassifier() 
XGBoost_model_sm.fit(X_train_sm_ss, y_train_sm)

# predict classification
y_test_pred=XGBoost_model_sm.predict(X_test_sm_ss)

# Score the train set
print(f'Train score: {XGBoost_model_sm.score(X_train_sm_ss, y_train_sm)*100:0.2f}%')

# Evaluate the test set
print(f'Test score: {XGBoost_model_sm.score(X_test_sm_ss, y_test)*100:0.2f}%')
print(f'Precision score: {precision_score(y_test,y_test_pred)*100:0.2f}%')
print(f'Recall score: {recall_score(y_test,y_test_pred)*100:0.2f}%')
print(f'F1 score: {f1_score(y_test,y_test_pred)*100:0.2f}%')

| Model              | Train Score | Test Score | Precision | Recall  | f1-score |
|--------------------|-------------|------------|-----------|---------|----------|
| XGBoost_model_sm   | 86.44%      | 82.90%     | 79.54%    | 73.07%  | 76.17%   |





The baseline XGBoost model is showing a ~83% accuracy score with slight overfitting with the train score being 86%. The model is providing higher precision than recall as well, with an f1-score of only 76%.


| Model              | Train Score | Test Score | Precision | Recall  | f1-score |
|--------------------|-------------|------------|-----------|---------|----------|
| log_reg_sm         | 79.90%      | 77.97%     | 72.92%    | 65.34%  | 68.92%   |
| random_forest_sm   | 99.08%      | 85.75%     | 83.57%    | 77.03%  | 80.16%   |
| XGBoost_model_sm   | 86.44%      | 82.90%     | 79.54%    | 73.07%  | 76.17%   |




Based on the three baseline models, the random_forest model has the highest test score and f1 score, however it has the highest overfitting with a 99% train score.  Next I will apply hyperparameters for each model to see if I can increase model performance.

# <a id='toc6_'></a>[5. Parameter Tuned Models](#toc0_)

## <a id='toc6_1_1_'></a>[5.1 Logistic Regression Hyperparameter Tuning](#toc0_)

### <a id='toc6_1_1_1_'></a>[5.1.1 Hyperparameter optimization for C](#toc0_)

In [ ]:
#Use Cross Validation to find the best C value

# Store the results
cross_validation_scores = []

# Create an array of powers of ten for the values of inverse regularization strength C
C_range = 10 ** np.array(range(-5, 5), dtype=float)

# Perform cross validation
for C in C_range:
    model = LogisticRegression(C=C, random_state=24, max_iter=1000)

    # The cross validation score (mean of scores from all folds)
    cv_score = np.mean(cross_val_score(model, X_train_sm_ss, y_train_sm, cv=5))

    # Append the score
    cross_validation_scores.append(cv_score)

plt.figure(figsize=(5,3))
plt.plot(C_range, cross_validation_scores, label="Cross Validation Score", marker=".")
plt.xscale("log")
plt.grid(alpha=0.1)
plt.xlabel("Regularization Parameter: C")
plt.ylabel("Cross Validation Score")
plt.title("Cross Validation Scores")
plt.legend()
plt.show()

# Print the optimal C
index_of_max = np.array(cross_validation_scores).argmax()
print("The best model has C = ", C_range[index_of_max])

By plotting CV scores of C, I can see that the CV score levels off between 0.1 and 1. C values above 1 are marginally increasing the CV score and therefore I will use C=1 for the next model.

### <a id='toc6_1_1_2_'></a>[5.1.2 Manual Hyperparameter Logistic Regression Model (log_reg_sm2)](#toc0_)

Optimized hyperparameter C with logistic regression

In [ ]:
# Instantiate the model
log_reg_sm2 = LogisticRegression(C=1, max_iter=1000)

# Perform 5-fold cross-validation for train
scores = cross_val_score(log_reg_sm2, X_train_sm_ss, y_train_sm, cv=5)
print(f"Cross-validation scores: {np.mean(scores)}")

# Fit the model
log_reg_sm2.fit(X_train_sm_ss, y_train_sm)

# predict classification
y_test_pred_sm2=log_reg_sm2.predict(X_test_sm_ss)

# Score the train set
print(f'Train score: {log_reg_sm2.score(X_train_sm_ss, y_train_sm)*100:0.2f}%')

# Evaluate the test set
print(f'Test score: {log_reg_sm2.score(X_test_sm_ss, y_test)*100:0.2f}%')

print(f'Precision score: {precision_score(y_test, y_test_pred_sm2)*100:0.2f}%')

print(f'Recall score: {recall_score(y_test, y_test_pred_sm2)*100:0.2f}%')

print(f'F1 score: {f1_score(y_test, y_test_pred_sm2)*100:0.2f}%')

The additional C values and max_iter did not seem to impact the model results. There were slight changes, but they were not significant. Next I will attemp PCA to see if there is any changes to the model performance.

| Model        | Train Score | Test Score | Precision | Recall | f1-score | Hyperparameter C | K-Fold Cross Val (k) | Max_Iter | PCA |
|--------------|-------------|------------|-----------|--------|----------|------------------|----------------------|----------|-----|
| log_reg_sm   | 79.90%      | 77.97%     | 72.92%    | 65.34% | 68.92%   | -                | -                    | -        | N   |
| log_reg_sm2  | 79.50%      | 77.94%     | 72.85%    | 65.38% | 68.91%   | C=1              | 5                    | 1000     | N   |


### <a id='toc6_1_1_3_'></a>[5.1.3 PCA with Logistic Regression model](#toc0_)
- Attempting Principal Component Analysis with the same hyperparameters as log_reg2 model to see if model scores improve by reducing the dimensionality, but keeping as much variation as possible.

In [ ]:
# Create the PCA object
PCA_object = PCA()

# Fit the PCA object to the data
PCA_object.fit(X_train_sm_ss)

# Transform the original data
X_PCA_train = PCA_object.transform(X_train_sm_ss)
X_PCA_test = PCA_object.transform(X_test_sm_ss)

In [ ]:
# Instantiate
log_reg_sm3 = LogisticRegression(C=1, max_iter=1000)

# Fit logreg model to X_train
log_reg_sm3.fit(X_PCA_train, y_train_sm)

# Scoring the logreg model
print(f' Train score: {log_reg_sm3.score(X_PCA_train, y_train_sm)}')
print(f' Test score: {log_reg_sm3.score(X_PCA_test, y_test)}')

# predict classification
y_test_pred_sm3 = log_reg_sm3.predict(X_PCA_test)

# Score the train set
print(f'Train score: {log_reg_sm3.score(X_PCA_train, y_train_sm)*100:0.2f}%')

# Evaluate the test set
print(f'Test score: {log_reg_sm3.score(X_PCA_test, y_test)*100:0.2f}%')

print(f'Precision score: {precision_score(y_test, y_test_pred_sm3)*100:0.2f}%')

print(f'Recall score: {recall_score(y_test, y_test_pred_sm3)*100:0.2f}%')

print(f'F1 score: {f1_score(y_test, y_test_pred_sm3)*100:0.2f}%')

After manually optimizing a C hyperparameter, K Fold Cross validation, and PCA, the overall accuracy of logistic regression did not improve. The f1-scores also remained the same.  

I will continue exploring the other two models as they had higher metrics and may provide more promising results. 

| Model        | Train Score | Test Score | Precision | Recall | f1-score | Hyperparameter C | K-Fold Cross Val (k) | Max_Iter | PCA |
|--------------|-------------|------------|-----------|--------|----------|------------------|----------------------|----------|-----|
| log_reg_sm   | 79.90%      | 77.97%     | 72.92%    | 65.34% | 68.92%   | -                | -                    | -        | N   |
| log_reg_sm2  | 79.50%      | 77.94%     | 72.85%    | 65.38% | 68.91%   | C=1              | 5                    | 1000     | N   |
| log_reg_sm3  | 79.90%      | 77.95%     | 72.86%    | 65.38% | 68.91%   | C=1              | -                    | 1000     | Y   |



## <a id='toc6_2_'></a>[5.2 Random Forest Hyperparameter Tuning](#toc0_)

Based on manual hyperparameter optimization (see appendix), I will run a new random forest model with the identified optimal hyperparameters.


### <a id='toc6_2_1_'></a>[5.2.1 Manual Hyperparameter Optimization Random Forest Model (random_forest_sm2)](#toc0_)

In [ ]:
# Instantiate random forest
random_forest_sm2 = RandomForestClassifier(n_estimators=1500, 
                                        max_depth=9, 
                                        min_samples_split=72, 
                                        min_samples_leaf=16,
                                        random_state=24,
                                        class_weight='balanced') 

# Fit the random forest model
random_forest_sm2.fit(X_train_sm_ss, y_train_sm)

# predict classification
y_test_pred_rf2=random_forest_sm2.predict(X_test_sm_ss)

# Score the train set
print(f'Train score: {random_forest_sm2.score(X_train_sm_ss, y_train_sm)*100:0.2f}%')

# Evaluate the test set
print(f'Test score: {random_forest_sm2.score(X_test_sm_ss, y_test)*100:0.2f}%')
print(f'Precision score: {precision_score(y_test,y_test_pred_rf2)*100:0.2f}%')
print(f'Recall score: {recall_score(y_test,y_test_pred_rf2)*100:0.2f}%')
print(f'F1 score: {f1_score(y_test,y_test_pred_rf2)*100:0.2f}%')

Random forest model with manual hyperparameters has decreased the overfitting significantly, however there is a 10% drop in f1-score where Recall decreased the most by 13%

| Model              | Train Score | Test Score | Precision | Recall  | f1-score | n_estimators | max_depth | min_samples_split | min_samples_leaf | class_weight |
|--------------------|-------------|------------|-----------|---------|----------|--------------|-----------|-------------------|------------------|--------------|
| random_forest_sm   | 99.08%      | 85.75%     | 83.57%    | 77.03%  | 80.16%   | 1000         | -         | -                 | -                | -            |
| random_forest_sm2  | 80.24%      | 79.80%     | 77.74%    | 64.44%  | 70.47%   | 1500         | 9         | 72                | 16               | balanced     |




### <a id='toc6_2_2_'></a>[5.2.2 Grid Searched HyperParameters Random Forest Model random_forest_sm3](#toc0_)
Grid Search Results for Best Hyperparameters for Random Forest (see apendix for details):
- max_depth: 11
- min_samples leaf: 1
- min_samples split: 60
- n_estimators: 1500

In [ ]:
# Instantiate random forest
random_forest_sm3 = RandomForestClassifier(n_estimators=1500, 
                                        max_depth=11, 
                                        min_samples_split=60, 
                                        min_samples_leaf=1,
                                        random_state=24,
                                        class_weight='balanced') 

# Fit the random forest model
random_forest_sm3.fit(X_train_sm_ss, y_train_sm)

# predict classification
y_test_pred_rf3=random_forest_sm3.predict(X_test_sm_ss)

# Score the train set
print(f'Train score: {random_forest_sm3.score(X_train_sm_ss, y_train_sm)*100:0.2f}%')

# Evaluate the test set
print(f'Test score: {random_forest_sm3.score(X_test_sm_ss, y_test)*100:0.2f}%')
print(f'Precision score: {precision_score(y_test,y_test_pred_rf3)*100:0.2f}%')
print(f'Recall score: {recall_score(y_test,y_test_pred_rf3)*100:0.2f}%')
print(f'F1 score: {f1_score(y_test,y_test_pred_rf3)*100:0.2f}%')

The random forest model with the gridsearched hyperparameters showed a 1% increase in accuracy and a ~2% increase in f1-score.

| Model              | Train Score | Test Score | Precision | Recall  | f1-score | n_estimators | max_depth | min_samples_split | min_samples_leaf | class_weight |
|--------------------|-------------|------------|-----------|---------|----------|--------------|-----------|-------------------|------------------|--------------|
| random_forest_sm   | 99.08%      | 85.75%     | 83.57%    | 77.03%  | 80.16%   | 1000         | -         | -                 | -                | -            |
| random_forest_sm2  | 80.24%      | 79.80%     | 77.74%    | 64.44%  | 70.47%   | 1500         | 9         | 72                | 16               | balanced     |
| random_forest_sm3  | 81.91%      | 80.80%     | 78.58%    | 66.88%  | 72.26%   | 1500         | 11        | 60                | 1                | balanced     |





The random forest model with the hyperparameters from the grid search are a lot more promising and are not overfitted.  I will take the random_forest2 model to evaluate the model further

## <a id='toc6_3_'></a>[5.3 XGBoost Model with Hyperparameter Tuning](#toc0_)

In [ ]:
# Instantiate XGBoost
XGBoost_model_sm2 = XGBClassifier(n_estimators= 400,
                                  learning_rate=0.1,
                                  max_depth=7,
                                  eval_metric='auc') 
XGBoost_model_sm2.fit(X_train_sm_ss, y_train_sm,
                      eval_set=[(X_test_sm_ss, y_test)],
                      verbose=True)

# predict classification
y_test_pred_xgb2=XGBoost_model_sm2.predict(X_test_sm_ss)

# Score the train set
print(f'Train score: {XGBoost_model_sm2.score(X_train_sm_ss, y_train_sm)*100:0.2f}%')

# Evaluate the test set
print(f'Test score: {XGBoost_model_sm2.score(X_test_sm_ss, y_test)*100:0.2f}%')
print(f'Precision score: {precision_score(y_test,y_test_pred_xgb2)*100:0.2f}%')
print(f'Recall score: {recall_score(y_test,y_test_pred_xgb2)*100:0.2f}%')
print(f'F1 score: {f1_score(y_test,y_test_pred_xgb2)*100:0.2f}%')

The XGBoost model with hyperparameters added increased the accuracy and f1-score only slightly by 1%. However, the model is now slightly more overfitted compared to the baseline model by ~1.5%.

| Model              | Train Score | Test Score | Precision | Recall  | f1-score | n_estimators | max_depth | learning_rate | eval_metric |
|--------------------|-------------|------------|-----------|---------|----------|--------------|-----------|---------------|-------------|
| XGBoost_model_sm   | 86.44%      | 82.90%     | 79.54%    | 73.07%  | 76.17%   | -            | -         | -             | -           |
| XGBoost_model_sm2  | 88.98%      | 83.86%     | 81.04%    | 74.20%  | 77.47%   | 400          | 7         | 0.1           | auc         |


### <a id='toc6_3_1_'></a>[5.3.1 Grid Searched Hyperparameter XGBoost Model XGBoost_model_sm3](#toc0_)
Grid Search Results for Best Hyperparameters for XGBoost (see apendix for details):
- n_estimators: 300
- max_depth: 7
- learning_rate: 0.2


In [ ]:
# Instantiate XGBoost
XGBoost_model_sm3 = XGBClassifier(n_estimators= 300,
                                  learning_rate=0.2,
                                  max_depth=7,
                                  eval_metric='auc') 
XGBoost_model_sm3.fit(X_train_sm_ss, y_train_sm,
                      eval_set=[(X_test_sm_ss, y_test)],
                      verbose=True)

# predict classification
y_test_pred_xgb3=XGBoost_model_sm3.predict(X_test_sm_ss)

# Score the train set
print(f'Train score: {XGBoost_model_sm3.score(X_train_sm_ss, y_train_sm)*100:0.2f}%')

# Evaluate the test set
print(f'Test score: {XGBoost_model_sm3.score(X_test_sm_ss, y_test)*100:0.2f}%')

print(f'Precision score: {precision_score(y_test, y_test_pred_xgb3)*100:0.2f}%')

print(f'Recall score: {recall_score(y_test, y_test_pred_xgb3)*100:0.2f}%')

print(f'F1 score: {f1_score(y_test, y_test_pred_xgb3)*100:0.2f}%')

The gridsearched hyperparameter model (XGBoost_model_sm3) resulted in another ~1% increase in f1-score similar to previous model run, however it is the most overfitted with the train score being almost 6.5% higher than test score.

| Model              | Train Score | Test Score | Precision | Recall  | f1-score | n_estimators | max_depth | learning_rate | eval_metric |
|--------------------|-------------|------------|-----------|---------|----------|--------------|-----------|---------------|-------------|
| XGBoost_model_sm   | 86.44%      | 82.90%     | 79.54%    | 73.07%  | 76.17%   |              |           |               |             |
| XGBoost_model_sm2  | 88.98%      | 83.86%     | 81.04%    | 74.20%  | 77.47%   | 400          | 7         | 0.1           | auc         |
| XGBoost_model_sm3  | 90.80%      | 84.33%     | 81.56%    | 75.07%  | 78.18%   | 300          | 7         | 0.2           | auc         |


# <a id='toc7_'></a>[6. Summary](#toc0_)

Before looking at further model evaluation, here is a summary table of the models run:


| Model         | Train Score | Test Score | Hyperparameter C | K-Fold Cross Val | Max Iter | PCA |
|---------------|-------------|------------|------------------|------------------|----------|-----|
| log_reg_sm    | 79.90%      | 77.97%     | -                |   -              | -        | N   |
| log_reg_sm2   | 79.50%      | n/a        | C=1              |   5              | 1000     | N   |
| log_reg_sm3   | 79.90%      | 77.94%     | C=1              |   -              | 1000     | Y   |

| Model              | Train Score | Test Score | Precision | Recall  | f1-score | n_estimators | max_depth | min_samples_split | min_samples_leaf | class_weight |
|--------------------|-------------|------------|-----------|---------|----------|--------------|-----------|-------------------|------------------|--------------|
| random_forest_sm   | 99.08%      | 85.75%     | 83.57%    | 77.03%  | 80.16%   | 1000         | -         | -                 | -                | -            |
| random_forest_sm2  | 80.24%      | 79.80%     | 77.74%    | 64.44%  | 70.47%   | 1500         | 9         | 72                | 16               | balanced     |
| random_forest_sm3  | 81.91%      | 80.80%     | 78.58%    | 66.88%  | 72.26%   | 1500         | 11        | 60                | 1                | balanced     |

| Model              | Train Score | Test Score | Precision | Recall  | f1-score | n_estimators | max_depth | learning_rate | eval_metric |
|--------------------|-------------|------------|-----------|---------|----------|--------------|-----------|---------------|-------------|
| XGBoost_model_sm   | 86.44%      | 82.90%     | 79.54%    | 73.07%  | 76.17%   |              |           |               |             |
| XGBoost_model_sm2  | 88.98%      | 83.86%     | 81.04%    | 74.20%  | 77.47%   | 400          | 7         | 0.1           | auc         |
| XGBoost_model_sm3  | 90.80%      | 84.33%     | 81.56%    | 75.07%  | 78.18%   | 300          | 7         | 0.2           | auc         |

Based on the three baseline models, the random_forest model has the highest test score and f1 score, however it has the highest overfitting with a 99% train score.  
- For logistic regression, optimizing the C value did not produce significant results. When applying principal component analysis to reduce dimensionality, I noticed that the accuracy and f1-scores also did not improve.
- For random forest, by manually optimizing hyperparameters, I was able to significantly decrease the overfitting. The train vs. test score accuracy difference decreased from 14% to less than 1%.  However, there was a 10% drop in the f1-score (with a 13% decrease in Recall). But when combining my manual optimization with grid search, I achieved a 3rd random forest model that maintained minimal overfitting and a 2% increase in f-1 score compared to my previous model run.
- For the XGBoost model, the results of adding hyperparameters were opposite of the Random Forest models. F1-score and accuracy displayed only a 1% increase, however it also increased overfitting. This pattern of increased overfitting continued when using gridsearch to further optimize the Hyperparameters.

For potential next iterative steps, I can return to feature engineering and remove stays in weekday nights and stay in weekend nights due to redundancy now that I have length of stay (los).

# <a id='toc8_'></a>[7. Export Data and Models](#toc0_)

Exporting models:

In [ ]:
# Pickle logistic regression models
joblib.dump(log_reg_sm, '../models/trained/log_reg_sm.pkl')

joblib.dump(log_reg_sm2, '../models/trained/log_reg_sm2.pkl')

joblib.dump(log_reg_sm3, '../models/trained/log_reg_sm3.pkl')

# Pickle random forest models
joblib.dump(random_forest_sm, '../models/trained/random_forest_sm.pkl')

joblib.dump(random_forest_sm2, '../models/trained/random_forest_sm2.pkl')

joblib.dump(random_forest_sm3, '../models/trained/random_forest_sm3.pkl')

# Pickle XGBoost models
joblib.dump(XGBoost_model_sm, '../models/trained/XGBoost_model_sm.pkl')

joblib.dump(XGBoost_model_sm2, '../models/trained/XGBoost_model_sm2.pkl')

joblib.dump(XGBoost_model_sm3, '../models/trained/XGBoost_model_sm3.pkl')


#pickle ss_sm scaler
joblib.dump(ss_sm, '../models/trained/scaler.pkl')

Exporting X/Y train and tests:


In [ ]:
joblib.dump(X_test, '../data/processed/X_test.joblib')
joblib.dump(X_test_sm_ss, '../data/processed/X_test_sm_ss.joblib')

joblib.dump(X_train, '../data/processed/X_train.joblib')
joblib.dump(X_train_sm, '../data/processed/X_train_sm.joblib')
joblib.dump(X_train_sm_ss, '../data/processed/X_train_sm_ss.joblib')

joblib.dump(y_test, '../data/processed/y_test.joblib')
joblib.dump(y_train_sm, '../data/processed/y_train_sm.joblib')

# <a id='toc9_'></a>[8. Appendix](#toc0_)

### <a id='toc9_1_1_'></a>[8.1 Random Forest Model: Manually identifying best hyperparameters](#toc0_)

In [ ]:
# Manually identify best max_depth for Random Forest

#Creating empty lists to keep track of scores
rf_cv_scores = []

# Create an array of max_depth
depth_range = np.array(range(1,15), dtype=int)

# Loop
for depth in depth_range:
    # Instantiate model
    rf_model = RandomForestClassifier(max_depth=depth)
   
    # Fit to TRAINING set
    cv_score = np.mean(cross_val_score(random_forest_sm, X_train_sm_ss, y_train_sm, cv=5))

    # Score on TRAINING set
    rf_cv_scores.append(cv_score)

#Plot cross validation result
plt.figure()
plt.plot(depth_range, rf_cv_scores, label="Cross Validation Score", marker=".")

#plt.grid(alpha=0.1)
plt.xlabel("Parameter: max_depth")
plt.ylabel("Cross Validation Score")
plt.title("max_depth Cross Validation Scores")
plt.legend()
plt.show()

# Print the optimal max_depth
index_of_max = np.array(rf_cv_scores).argmax()
print("The best model has max_depth = ", depth_range[index_of_max])

In [ ]:
# Manually identify best min_samples_leaf for random forests

#Creating empty lists to keep track of csores
rf_cv_scores = []

# Create an array of min_samples_leaf
min_samples = np.array(range(1,30,5), dtype=int)

# Loop
for leaf in min_samples:
    # Instantiate model
    rf_model = RandomForestClassifier(min_samples_leaf=leaf)
   
    # Fit to TRAINING set
    cv_score = np.mean(cross_val_score(random_forest_sm, X_train_sm_ss, y_train_sm, cv=5))

    # Score on TRAINING set
    rf_cv_scores.append(cv_score)

#Plot cross validation result
plt.figure()
plt.plot(min_samples, rf_cv_scores, label="Cross Validation Score", marker=".")

#plt.grid(alpha=0.1)
plt.xlabel("Parameter: min_samples_leaf")
plt.ylabel("Cross Validation Score")
plt.title("min_samples_leaf Cross Validation Scores")
plt.legend()
plt.show()

# Print the optimal min_samples_leaf
index_of_max = np.array(rf_cv_scores).argmax()
print("The best model has min_samples_leaf = ", min_samples[index_of_max])

In [ ]:
# Manually identify best min_samples_split

#Creating empty lists to keep track of csores
rf_cv_scores = []

# Create an array of min_samples_split
min_splits = np.array(range(2,100,5), dtype=int)

# Loop
for split in min_splits:
    # Instantiate model
    rf_model = RandomForestClassifier(min_samples_split=split)
   
    # Fit to TRAINING set
    cv_score = np.mean(cross_val_score(random_forest_sm, X_train_sm_ss, y_train_sm, cv=5))

    # Score on TRAINING set
    rf_cv_scores.append(cv_score)

#Plot cross validation result
plt.figure()
plt.plot(min_splits, rf_cv_scores, label="Cross Validation Score", marker=".")

#plt.grid(alpha=0.1)
plt.xlabel("Parameter: min_samples_split")
plt.ylabel("Cross Validation Score")
plt.title("min_samples_split Cross Validation Scores")
plt.legend()
plt.show()

# Print the optimal min_samples_split
index_of_max = np.array(rf_cv_scores).argmax()
print("The best model has min_samples_split = ", min_splits[index_of_max])

In [ ]:
# Manually identify best n_estimators

#Creating empty lists to keep track of csores
rf_cv_scores = []

# Create an array of n_estimators
n_estimators = [100, 500, 1000, 1500]

# Loop
for n in n_estimators:
    # Instantiate model
    rf_model = RandomForestClassifier(n_estimators=n)
   
    # Fit to TRAINING set
    cv_score = np.mean(cross_val_score(random_forest_sm, X_train_sm_ss, y_train_sm, cv=5))

    # Score on TRAINING set
    rf_cv_scores.append(cv_score)

#Plot cross validation result
plt.figure()
plt.plot(n_estimators, rf_cv_scores, label="Cross Validation Score", marker=".")

#plt.grid(alpha=0.1)
plt.xlabel("Parameter: n_estimators")
plt.ylabel("Cross Validation Score")
plt.title("n_estimators Cross Validation Scores")
plt.legend()
plt.show()

# Print the optimal C
index_of_max = np.array(rf_cv_scores).argmax()
print("The best model has n_estimators = ", n_estimators[index_of_max])

## <a id='toc9_2_'></a>[8.2 Grid Searches](#toc0_)

### <a id='toc9_2_1_'></a>[8.2.1 Random Forest Grid Search for Hyperparameters](#toc0_)


In [ ]:
#Creating pipeline for grid search
rf_estimators = [('random_forest', RandomForestClassifier())]
rf_pipe = Pipeline(rf_estimators)

#hyperparameters to search
rf_params = {'random_forest__n_estimators': [200, 1500], # testing with just 100 and 200 first to prevent overfitting
             'random_forest__max_depth': [9, 11],
             'random_forest__min_samples_split': [72, 60],
             'random_forest__min_samples_leaf': [1, 16]}

rf_grid_search = GridSearchCV(rf_pipe, 
                             param_grid=rf_params,
                             cv = 3,
                             verbose=2,#show progress of the grid search
                             n_jobs=-1) #parallel computations and use multiple cores 

rf_fitted_search = rf_grid_search.fit(X_train_sm_ss, y_train_sm)

In [ ]:
# Save grid search results (Uncomment to code below to save grid search)
#joblib.dump(rf_fitted_search, '../models/rf_fitted_search.pkl')

# reload grid search results (Uncomment the code below to load grid search)
#rf_fitted_search = joblib.load('../models/rf_fitted_search.pkl')


In [ ]:
print("Best Random Forest Parameters:", rf_fitted_search.best_params_)
print("Best Cross-Validation Accuracy:", rf_fitted_search.best_score_)

### <a id='toc9_2_2_'></a>[8.2.2 XGBoost Model Grid Search for optimial Hyperparameters](#toc0_)


In [ ]:
#Creating pipeline for grid search
XGB_estimators = [('XGBoost', XGBClassifier(eval_metric='auc'))]
XGB_pipe = Pipeline(XGB_estimators)

#hyperparameters to search
XGB_params = {'XGBoost__n_estimators': [100, 200, 300], 
             'XGBoost__max_depth': [3, 5, 7],
             'XGBoost__learning_rate': [0.01, 0.1, 0.2]}

XGB_grid_search = GridSearchCV(XGB_pipe, 
                              param_grid=XGB_params,
                              scoring='accuracy',
                              cv = 3,
                              verbose=2, #show progress of the grid search
                              n_jobs=-1) #parallel computations and use multiple cores 

XGB_fitted_search = XGB_grid_search.fit(X_train_sm_ss, y_train_sm)

In [ ]:
# Save grid search results (Uncomment to code below to save grid search)
#joblib.dump(XGB_fitted_search, '../models/XGB_fitted_search.pkl')

# reload grid search results (Uncomment the code below to load grid search)
#rf_fitted_search = joblib.load('../models/XGB_fitted_search.pkl')


In [ ]:
print("Best Random Forest Parameters:", XGB_fitted_search.best_params_)
print("Best Cross-Validation Accuracy:", XGB_fitted_search.best_score_)